In [1]:
import optuna
import torch
import torch.nn as nn
from torchinfo import summary
from torch.utils.data import DataLoader,Dataset
import pandas as pd
from sklearn.model_selection import train_test_split
import torch.optim as optim
import matplotlib.pyplot as plt

In [2]:
torch.manual_seed(42)
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"using {device}")

using cuda


In [3]:
df_train=pd.read_csv(r'C:\Coding\ML_DL\Datasets\fashion-mnist_train.csv')
df_test=pd.read_csv(r'C:\Coding\ML_DL\Datasets\fashion-mnist_test.csv')


In [4]:
X_train=df_train.iloc[:,1:]
X_test=df_test.iloc[:,1:]
y_train=df_train.iloc[:,0]
y_test=df_test.iloc[:,0]


X_train = X_train.reset_index(drop=True)
y_train = y_train.reset_index(drop=True)

X_test  = X_test.reset_index(drop=True)
y_test  = y_test.reset_index(drop=True)

X_train=X_train/255
X_test=X_test/255

In [5]:
class CustomDataset(Dataset):
    def __init__(self, features, labels):
        self.features = torch.tensor(features.values, dtype=torch.float32)
        self.labels = torch.tensor(labels.values, dtype=torch.long)

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]
    

train_dataset=CustomDataset(X_train,y_train)
test_dataset=CustomDataset(X_test,y_test)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True,pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False,pin_memory=True)


In [6]:
class NeuralNet(nn.Module):
    def __init__(self, input_dim, output_dim, num_hidden_layer, neurons_per_layer,dropout_rate):
        super().__init__()

        layers = []

        for _ in range(num_hidden_layer):
            layers.append(nn.Linear(input_dim, neurons_per_layer))
            layers.append(nn.BatchNorm1d(neurons_per_layer))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout_rate))

            input_dim = neurons_per_layer  

        # Output layer
        layers.append(nn.Linear(neurons_per_layer, output_dim))

        self.model = nn.Sequential(*layers)

    def forward(self, X):
        return self.model(X)


In [7]:
def objective(trial):

    #extract hp values for next trial from search space
    num_hidden_layers=trial.suggest_int("num_hidden_layer",1,5)
    neurons_per_layer=trial.suggest_int("neurons",8,128,step=8)
    epochs=trial.suggest_int('epochs',10,50,step=5)
    learning_rate=trial.suggest_float('lr',1e-5,1e-1,log=True)
    dropout_rate=trial.suggest_float('dropout layer',0.1,0.5,step=0.1)




    #model init
    input_dim=784
    output_dim=10
    model=NeuralNet(input_dim,output_dim,num_hidden_layers,neurons_per_layer,dropout_rate)
    model.to(device)

    #optim selection
    optimizer=optim.SGD(model.parameters(),lr=learning_rate,weight_decay=1e-4)

    #set loss
    criterion=nn.CrossEntropyLoss()
    #training loop
    for epoch in range(epochs):
    
        for batch_features,batch_labels in train_loader:
            batch_features,batch_labels=batch_features.to(device), batch_labels.to(device)
            #forward prop
            output=model(batch_features)

            #calculate loss
            loss=criterion(output,batch_labels)

            #back prop
            optimizer.zero_grad()
            loss.backward()

            #update params
            optimizer.step()


    model.eval()
    total = 0
    correct = 0

    with torch.no_grad():

        for batch_features, batch_labels in test_loader:
            batch_features,batch_labels=batch_features.to(device), batch_labels.to(device)

            outputs = model(batch_features)

            _, predicted = torch.max(outputs, 1)

            total = total + batch_labels.shape[0]

            correct = correct + (predicted == batch_labels).sum().item()

            accuracy=(correct/total)


    return accuracy


In [8]:
study=optuna.create_study(directions=['maximize'])
study.optimize(objective,n_trials=10)

[I 2026-01-22 00:40:24,850] A new study created in memory with name: no-name-5a099cfa-6d4c-4538-8974-f4b0b530cabb
[I 2026-01-22 00:45:16,853] Trial 0 finished with value: 0.8754 and parameters: {'num_hidden_layer': 5, 'neurons': 64, 'epochs': 30, 'lr': 0.0010410636553303597, 'dropout layer': 0.2}. Best is trial 0 with value: 0.8754.
[I 2026-01-22 00:50:25,263] Trial 1 finished with value: 0.8296 and parameters: {'num_hidden_layer': 2, 'neurons': 72, 'epochs': 50, 'lr': 0.00012044347167822942, 'dropout layer': 0.4}. Best is trial 0 with value: 0.8754.
[I 2026-01-22 00:52:24,291] Trial 2 finished with value: 0.8403 and parameters: {'num_hidden_layer': 4, 'neurons': 48, 'epochs': 15, 'lr': 0.0003653651617562883, 'dropout layer': 0.1}. Best is trial 0 with value: 0.8754.
[I 2026-01-22 00:57:13,728] Trial 3 finished with value: 0.891 and parameters: {'num_hidden_layer': 5, 'neurons': 88, 'epochs': 30, 'lr': 0.042860913367250335, 'dropout layer': 0.2}. Best is trial 3 with value: 0.891.
[I 2

In [9]:
study.best_value

0.891

In [10]:
study.best_params

{'num_hidden_layer': 5,
 'neurons': 88,
 'epochs': 30,
 'lr': 0.042860913367250335,
 'dropout layer': 0.2}